# ⚽ ¿Quién será campeón del Mundial 2026?
### Un modelo predictivo con Elo + Poisson + simulación de Monte Carlo

**Mundial FIFA 2026 — USA / Canadá / México** · 48 selecciones · arranca el 11 de junio de 2026.

La idea es estimar, **antes de que ruede la pelota**, la probabilidad de que cada selección
levante la copa. Lo hacemos en tres capas:

| Capa | Qué hace | Técnica |
|------|----------|---------|
| **1. Rating Elo** | Mide la "fuerza" de cada selección usando todo el historial (1872–2026) | Elo estilo *eloratings.net* |
| **2. Modelo de goles** | Traduce la diferencia de Elo + localía en goles esperados | Regresión de Poisson |
| **3. Simulación** | Juega el Mundial completo miles de veces y cuenta campeones | Monte Carlo |

> **Datos:** dataset público [martj42/international_results](https://github.com/martj42/international_results),
> que además ya trae el **fixture oficial del Mundial 2026** → de ahí reconstruimos los 12 grupos exactos.

---

## 0 · Preparación
Toda la lógica vive en `mundial2026.py`; aquí la usamos paso a paso y explicamos qué hace.

In [ ]:
import pandas as pd, numpy as np
import mundial2026 as M     # nuestro módulo
pd.set_option('display.max_rows', 60)

df = pd.read_csv('data/results.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
jugados = df.dropna(subset=['home_score','away_score'])
print(f"{len(jugados):,} partidos jugados, de {jugados.date.min().date()} a {jugados.date.max().date()}")
jugados.tail(3)

## 1 · Rating Elo

El Elo asigna a cada selección un número de "fuerza". Tras cada partido:

$$R_{nuevo} = R_{viejo} + K\,(W - W_e), \qquad W_e = \frac{1}{1 + 10^{-dr/400}}$$

- $W$ = resultado real (1 gana / 0.5 empata / 0 pierde)
- $W_e$ = resultado esperado según la diferencia de rating $dr$ (incluye +100 de localía)
- $K$ = importancia del partido (Mundial=60, clasificatorias=40, amistoso=20…) amplificada si hay goleada.

Un punto clave: **arrastra la forma reciente**. España llega como campeona de la Euro 2024 y eso se refleja en su Elo.

In [ ]:
elos, hist = M.calcular_elo(jugados)

mundialistas = [t for eqs in M.GRUPOS.values() for t in eqs]
ranking = (pd.Series({t: elos[t] for t in mundialistas})
             .sort_values(ascending=False).round(0).astype(int))
ranking.head(15).to_frame('Elo')

## 2 · Modelo de goles (Poisson)

El Elo nos da quién es más fuerte, pero para simular necesitamos **goles** (para la diferencia de gol en los grupos).
Ajustamos una regresión de Poisson sobre todos los partidos desde 1960:

$$\log(\text{goles esperados}) = \beta_0 + \beta_1\,(\text{Elo}_{propio}-\text{Elo}_{rival}) + \beta_2\,\text{localía}$$

In [ ]:
modelo = M.entrenar_modelo_goles(hist)
print(modelo.summary().tables[1])
b = modelo.params
print(f"\nLocalía: un equipo de local marca {np.exp(b['local']):.2f}x goles.")
print(f"Ventaja de 100 pts de Elo: {np.exp(b['elo_diff']*100):.2f}x goles.")

**Ejemplo:** goles esperados en un España (Elo alto) vs. un rival medio, en cancha neutral:

In [ ]:
la, lb = M.lambdas(modelo, elos['Spain'], elos['Jordan'])
print(f"Spain {la:.2f}  -  {lb:.2f} Jordan   (goles esperados)")

## 3 · Estructura del torneo

Reconstruida desde el fixture: **12 grupos de 4**. Clasifican 1° y 2° de cada grupo + los **8 mejores terceros**
= 32 equipos a dieciseisavos. El bracket de eliminatorias sigue el formato **oficial de FIFA** (códigos `R32`, `R16`, …).

In [ ]:
for g, eqs in M.GRUPOS.items():
    print(f"Grupo {g}: " + ", ".join(eqs))

Simulemos **un** grupo para ver cómo queda la tabla (puntos, dif. de gol, goles):

In [ ]:
tabla = M.jugar_grupo(modelo, elos, 'E', M.GRUPOS['E'])
pd.DataFrame(tabla, columns=['Equipo','Pts','DifGol','GF','desempate']).drop(columns='desempate')

## 4 · Monte Carlo: jugar el Mundial miles de veces

Cada simulación juega los 12 grupos, ordena los terceros, arma el bracket y resuelve todas las
eliminatorias (en mata-mata, si hay empate se define por probabilidad Elo = "penales").
Repetimos **20.000 veces** y contamos.

> Tarda ~2–3 min. Si quieres una prueba rápida, baja `n_sims` a 3000.

In [ ]:
res = M.monte_carlo(modelo, elos, n_sims=20000)
res.head(15).style.format({'P_campeon':'{:.1%}','P_final':'{:.1%}',
                           'P_semi':'{:.1%}','P_octavos':'{:.1%}'})

## 5 · Resultado: los favoritos al título

In [ ]:
import matplotlib.pyplot as plt
top = res.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(top.equipo, top.P_campeon*100, color='#2a7ae2')
for y,v in enumerate(top.P_campeon*100): ax.text(v+0.15,y,f'{v:.1f}%',va='center')
ax.set_xlabel('Probabilidad de ser campeón (%)')
ax.set_title('Mundial 2026 — Favoritos al título')
ax.margins(x=0.12); plt.tight_layout(); plt.show()

## 6 · Conclusiones y limitaciones

**Lectura del modelo:** un puñado de selecciones concentra casi todas las opciones de título; el resto
funciona como "tapados". Las probabilidades reflejan **fuerza histórica + forma reciente vía Elo**, no lesiones,
convocatorias ni el sorteo emocional del momento.

**Limitaciones / ideas para mejorar:**
- El Elo no "sabe" de lesiones, bajas ni cambios de DT recientes.
- No modelamos la ventaja de los **anfitriones** en fases finales (solo en sus partidos de grupo).
- La asignación de los 8 terceros usa un emparejamiento válido, no la tabla exacta de FIFA (efecto mínimo).
- Mejoras posibles: Poisson bivariado (correlación de goles), rating ofensivo/defensivo separado,
  ponderar más los partidos recientes, o incorporar el valor de mercado de los planteles.

*Proyecto personal — hecho por diversión.* ⚽